In [2]:
import torch

from cryosiam.data_structure.dense_simsiam_config import load_dense_simsiam_config
from cryosiam.networks.nets import DenseSimSiam

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [3]:
path = "/g/kreshuk/talks/cryosiam/cryosiam/apps/dense_simsiam_pretraining/config.yaml"
cfg = load_dense_simsiam_config(path)

In [3]:
net = DenseSimSiam(
    block_type=cfg.parameters.network.block_type,
    spatial_dims=cfg.parameters.network.spatial_dims,
    n_input_channels=cfg.parameters.network.in_channels,
    num_layers=cfg.parameters.network.num_layers,
    num_filters=cfg.parameters.network.num_filters,
    fpn_channels=cfg.parameters.network.fpn_channels,
    no_max_pool=cfg.parameters.network.no_max_pool,
    dim=cfg.parameters.network.dim,
    pred_dim=cfg.parameters.network.pred_dim,
    dense_dim=cfg.parameters.network.dense_dim,
    dense_pred_dim=cfg.parameters.network.dense_pred_dim,
    include_levels=cfg.parameters.network.include_levels_loss,
    add_later_conv=cfg.parameters.network.add_fpn_later_conv,
    decoder_type=cfg.parameters.network.decoder_type,
    decoder_layers=cfg.parameters.network.fpn_layers
)

In [4]:
# count number of parameters in model and dtype of parameters
num_params = sum(p.numel() for p in net.parameters())
print(f"Number of parameters: {num_params}")

Number of parameters: 15345088


In [5]:
# dtype of network parameters
dtype = next(net.parameters()).dtype
print(f"Parameter dtype: {dtype}")

Parameter dtype: torch.float32


In [6]:
# Memory estimate for training
import torch

B = 10
num_views = 2
C = cfg.parameters.network.in_channels
D, H, W = cfg.parameters.data.view_size

# bytes per scalar from parameter dtype (typically float32 -> 4 bytes)
bytes_per = torch.tensor([], dtype=dtype).element_size()

# 1) Raw input tensor memory (one forward input batch)
n_input_elements = B * num_views * C * D * H * W
input_mem_gb = n_input_elements * bytes_per / (1024**3)

# 2) Parameters, gradients, and SGD momentum state
param_mem_gb = num_params * bytes_per / (1024**3)
grad_mem_gb = param_mem_gb
sgd_momentum_mem_gb = param_mem_gb  # one momentum buffer per parameter

# 3) Persistent training memory (without activations)
persistent_mem_gb = param_mem_gb + grad_mem_gb + sgd_momentum_mem_gb

# 4) Activation memory is architecture-dependent; use a multiplier on input memory
act_factor_low = 20
act_factor_high = 60
activation_low_gb = input_mem_gb * act_factor_low
activation_high_gb = input_mem_gb * act_factor_high

total_low_gb = persistent_mem_gb + activation_low_gb
total_high_gb = persistent_mem_gb + activation_high_gb

print(f"num_params                : {num_params:,}")
print(f"dtype                     : {dtype} ({bytes_per} bytes)")
print(f"input memory              : {input_mem_gb:.3f} GB")
print(f"persistent (params+grads+momentum): {persistent_mem_gb:.3f} GB")
print(f"activation estimate       : {activation_low_gb:.3f} - {activation_high_gb:.3f} GB")
print(f"total training estimate   : {total_low_gb:.3f} - {total_high_gb:.3f} GB")

# Optional practical headroom
print(f"with 20% headroom         : {1.2*total_low_gb:.3f} - {1.2*total_high_gb:.3f} GB")

num_params                : 15,345,088
dtype                     : torch.float32 (4 bytes)
input memory              : 0.020 GB
persistent (params+grads+momentum): 0.171 GB
activation estimate       : 0.391 - 1.172 GB
total training estimate   : 0.562 - 1.343 GB
with 20% headroom         : 0.675 - 1.612 GB


In [9]:
# Empirical GPU memory sweep for one train step (if CUDA is available)
import gc
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    net_cuda = net.to(device).train()

    # B=1 is invalid here due BatchNorm in training mode for global branch.
    test_batches = [2, 3, 4, 6, 8, 10]
    results = []

    for b in test_batches:
        try:
            optimizer = torch.optim.SGD(net_cuda.parameters(), lr=0.5, momentum=0.9, weight_decay=1e-5)
            x1 = torch.randn(b, C, D, H, W, device=device, dtype=dtype)
            x2 = torch.randn(b, C, D, H, W, device=device, dtype=dtype)

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)

            optimizer.zero_grad(set_to_none=True)
            out = net_cuda(x1, x2)

            if isinstance(out, dict):
                loss = sum(v.float().mean() for v in out.values() if torch.is_tensor(v))
            elif isinstance(out, (tuple, list)):
                loss = sum(v.float().mean() for v in out if torch.is_tensor(v))
            elif torch.is_tensor(out):
                loss = out.float().mean()
            else:
                raise RuntimeError(f"Unexpected network output type: {type(out)}")

            loss.backward()
            optimizer.step()
            torch.cuda.synchronize(device)

            peak_alloc_gb = torch.cuda.max_memory_allocated(device) / (1024**3)
            peak_reserved_gb = torch.cuda.max_memory_reserved(device) / (1024**3)
            results.append((b, peak_alloc_gb, peak_reserved_gb, True))
            print(f"B={b:2d}: peak allocated={peak_alloc_gb:.3f} GB, peak reserved={peak_reserved_gb:.3f} GB")

        except torch.cuda.OutOfMemoryError:
            results.append((b, float('nan'), float('nan'), False))
            print(f"B={b:2d}: OOM")
            torch.cuda.empty_cache()

        finally:
            del optimizer
            gc.collect()
            torch.cuda.empty_cache()

    ok = [(b, a) for (b, a, _, success) in results if success]
    if len(ok) >= 2:
        b1, m1 = ok[-2]
        b2, m2 = ok[-1]
        slope = (m2 - m1) / (b2 - b1)
        est_b10 = m2 + slope * (10 - b2)
        print(f"Linear estimate peak allocated at B=10: {est_b10:.3f} GB")
else:
    print("CUDA not available; cannot measure empirical GPU peak memory.")

B= 2: peak allocated=7.420 GB, peak reserved=7.686 GB
B= 3: OOM
B= 4: OOM
B= 6: OOM
B= 8: OOM
B=10: OOM


In [10]:
# Structured memory estimator for DenseSimSiam pretraining
import math
import torch

GiB = 1024**3
bytes_per = torch.tensor([], dtype=dtype).element_size()

# Config shortcuts
view_size = list(cfg.parameters.data.view_size)
num_views = 2
in_ch = cfg.parameters.network.in_channels
num_filters = list(cfg.parameters.network.num_filters)
fpn_ch = int(cfg.parameters.network.fpn_channels)
dense_dim = int(cfg.parameters.network.dense_dim)
include_levels = bool(cfg.parameters.network.include_levels_loss)
overlap = float(cfg.parameters.data.view_overlap)

if len(view_size) != 3:
    raise ValueError(f"This estimator currently assumes 3D views, got {view_size}")

def tensor_gib(numel: int, bytes_scalar: int = bytes_per) -> float:
    return (numel * bytes_scalar) / GiB

def vol(shape):
    return int(math.prod(shape))

def downsample_shape(shape, factor):
    return [max(1, s // factor) for s in shape]

def explicit_dynamic_gib(batch_size: int) -> tuple[dict, float]:
    # Input tensors used by the model forward (x1, x2).
    input_gib = tensor_gib(batch_size * num_views * in_ch * vol(view_size))

    # Encoder outputs retained for backward for both views: res2,res3,res4,res5
    enc_shapes = [
        downsample_shape(view_size, 1),   # res2
        downsample_shape(view_size, 2),   # res3
        downsample_shape(view_size, 4),   # res4
        downsample_shape(view_size, 8),   # res5
    ]
    encoder_gib = 0.0
    for ch, shp in zip(num_filters, enc_shapes):
        encoder_gib += tensor_gib(batch_size * num_views * ch * vol(shp))

    # Decoder outputs (o4,o3,o2,o1), all with fpn_ch channels for both views.
    dec_shapes = [
        downsample_shape(view_size, 1),   # o4
        downsample_shape(view_size, 2),   # o3
        downsample_shape(view_size, 4),   # o2
        downsample_shape(view_size, 8),   # o1
    ]
    decoder_gib = sum(
        tensor_gib(batch_size * num_views * fpn_ch * vol(shp)) for shp in dec_shapes
    )

    # Main dense projector/predictor outputs at full decoder resolution (z1,z2,p1,p2).
    main_head_gib = 2.0 * tensor_gib(batch_size * num_views * dense_dim * vol(dec_shapes[0]))

    # Optional level heads at o3,o2,o1 (z and p for each level, both views).
    level_head_gib = 0.0
    if include_levels:
        for shp in dec_shapes[1:]:
            level_head_gib += 2.0 * tensor_gib(batch_size * num_views * dense_dim * vol(shp))

    # Overlap-crop copies in select_overlap_pixels for dense + level losses.
    overlap_shape = [
        max(1, int(round(s * (overlap ** (1.0 / 3.0))))) for s in view_size
    ]
    overlap_shape = [min(s, o) for s, o in zip(view_size, overlap_shape)]
    dense_loss_copies_gib = 4.0 * tensor_gib(batch_size * dense_dim * vol(overlap_shape))

    level_loss_copies_gib = 0.0
    if include_levels:
        curr = overlap_shape[:]
        for _ in range(3):
            curr = downsample_shape(curr, 2)
            level_loss_copies_gib += 4.0 * tensor_gib(batch_size * dense_dim * vol(curr))

    components = {
        "inputs": input_gib,
        "encoder": encoder_gib,
        "decoder": decoder_gib,
        "main_heads": main_head_gib,
        "level_heads": level_head_gib,
        "loss_copies": dense_loss_copies_gib + level_loss_copies_gib,
    }
    dynamic = sum(components.values())
    return components, dynamic

# Persistent memory (optimizer state + gradients + parameters for SGD+momentum).
param_gib = tensor_gib(num_params)
persistent_gib = 3.0 * param_gib

# Try to calibrate with empirical measurement from cell 7 if available.
cal_b = None
cal_peak = None
if "results" in globals() and isinstance(results, list):
    ok = [(b, alloc) for (b, alloc, _reserved, success) in results if success]
    if ok:
        cal_b, cal_peak = ok[0]

# Fallback to known measured point from this notebook session.
if cal_b is None or cal_peak is None:
    cal_b, cal_peak = 2, 7.420

_, dyn_cal = explicit_dynamic_gib(cal_b)
alpha = max(1.0, (cal_peak - persistent_gib) / max(dyn_cal, 1e-9))

# Report at configured batch size.
B_cfg = int(cfg.hyper_parameters.batch_size)
comp_cfg, dyn_cfg = explicit_dynamic_gib(B_cfg)
est_cfg = persistent_gib + alpha * dyn_cfg

print("=== Memory Estimator Summary ===")
print(f"dtype bytes/element                  : {bytes_per}")
print(f"persistent (params+grads+momentum)   : {persistent_gib:.3f} GiB")
print(f"calibration point                    : B={cal_b}, peak_alloc={cal_peak:.3f} GiB")
print(f"calibration factor alpha             : {alpha:.2f}")
print(f"configured batch size                : B={B_cfg}")
print("--- explicit dynamic components at configured B (before alpha) ---")
for k, v in comp_cfg.items():
    print(f"{k:34s}: {v:.3f} GiB")
print(f"dynamic subtotal                     : {dyn_cfg:.3f} GiB")
print(f"estimated total (calibrated)         : {est_cfg:.3f} GiB")

print("\n=== Estimated Peak Allocated by Batch Size ===")
for b in [1, 2, 3, 4, 6, 8, 10]:
    _c, dyn = explicit_dynamic_gib(b)
    est = persistent_gib + alpha * dyn
    print(f"B={b:2d} -> estimated peak allocated: {est:.3f} GiB")

=== Memory Estimator Summary ===
dtype bytes/element                  : 4
persistent (params+grads+momentum)   : 0.171 GiB
calibration point                    : B=2, peak_alloc=7.420 GiB
calibration factor alpha             : 4.11
configured batch size                : B=10
--- explicit dynamic components at configured B (before alpha) ---
inputs                            : 0.020 GiB
encoder                           : 1.660 GiB
decoder                           : 2.856 GiB
main_heads                        : 2.500 GiB
level_heads                       : 0.356 GiB
loss_copies                       : 1.433 GiB
dynamic subtotal                     : 8.825 GiB
estimated total (calibrated)         : 36.415 GiB

=== Estimated Peak Allocated by Batch Size ===
B= 1 -> estimated peak allocated: 3.796 GiB
B= 2 -> estimated peak allocated: 7.420 GiB
B= 3 -> estimated peak allocated: 11.045 GiB
B= 4 -> estimated peak allocated: 14.669 GiB
B= 6 -> estimated peak allocated: 21.918 GiB
B= 8 -> est

In [4]:
# CPU memory estimator for data loading and preprocessing
import math

GiB = 1024**3
bytes_per = 4  # float32

# Data config
patch_size = list(cfg.parameters.data.patch_size)
view_size = list(cfg.parameters.data.view_size)
batch_size = int(cfg.hyper_parameters.batch_size)
cache_rate = float(cfg.hyper_parameters.cache_rate) if hasattr(cfg.hyper_parameters, 'cache_rate') else 0.0
num_workers = 10  # typical from module.py train_dataloader
pin_memory = True

def vol(shape):
    return int(math.prod(shape))

# Estimate number of training samples (rough based on config)
# If you know actual dataset size, update this
estimated_total_samples = 100  # placeholder; adjust to your actual dataset size

print('=== CPU Memory Estimator for Data Loading ===' )
print(f'batch_size                           : {batch_size}')
print(f'patch_size (per sample)              : {patch_size}')
print(f'view_size (per forward)              : {view_size}')
print(f'num_workers (DataLoader)              : {num_workers}')
print(f'pin_memory                           : {pin_memory}')
print(f'cache_rate                          : {cache_rate}')
print()

# 1. Input batch tensor in CPU memory before GPU transfer
batch_patch_bytes = batch_size * 1 * vol(patch_size) * bytes_per
batch_patch_gib = batch_patch_bytes / GiB
print(f'--- One full batch in CPU (before GPU transfer) ---')
print(f'one patch batch (64x64x64 view)     : {batch_patch_gib:.3f} GiB')
print()

# 2. DataLoader worker memory
# Each worker loads and preprocesses samples independently
# Estimate: 1 sample in memory per worker during loading
worker_memory_per_sample_gib = (1 * 1 * vol(patch_size) * bytes_per) / GiB
total_worker_memory_gib = num_workers * worker_memory_per_sample_gib
print(f'--- DataLoader worker buffers ---')
print(f'memory per worker (1 sample)         : {worker_memory_per_sample_gib:.3f} GiB')
print(f'total {num_workers} workers                     : {total_worker_memory_gib:.3f} GiB')
print()

# 3. Pinned memory buffers
# Pin pool typically = batch_size * 2-4 batches for prefetch
pinned_factor = 2 if pin_memory else 0
pinned_memory_gib = batch_patch_gib * pinned_factor
print(f'--- Pinned memory for async GPU transfer ---')
print(f'pinned buffer size ({pinned_factor}x batch)         : {pinned_memory_gib:.3f} GiB')
print()

# 4. CacheDataset memory (if enabled)
cache_memory_gib = 0.0
if cache_rate > 0:
    cached_samples = int(estimated_total_samples * cache_rate)
    cache_memory_gib = (cached_samples * 1 * vol(patch_size) * bytes_per) / GiB
    print(f'--- CacheDataset (if enabled) ---')
    print(f'cache_rate                          : {cache_rate}')
    print(f'estimated cached samples            : {cached_samples}')
    print(f'total cache memory                  : {cache_memory_gib:.3f} GiB')
    print()
else:
    print(f'--- CacheDataset ---')
    print(f'cache_rate=0 (streaming mode, no caching)')
    print()

# 5. Transform and preprocessing overhead
# Rough estimate: 2x batch size for intermediate processing
transform_overhead_gib = batch_patch_gib * 2
print(f'--- Transform pipeline overhead ---')
print(f'estimated intermediate tensors      : {transform_overhead_gib:.3f} GiB')
print()

# Total CPU memory estimate
total_cpu_base = batch_patch_gib + total_worker_memory_gib + pinned_memory_gib + transform_overhead_gib
total_cpu_with_cache = total_cpu_base + cache_memory_gib

print('=== CPU Memory Summary ===')
print(f'base (batch+workers+pinned+xforms)  : {total_cpu_base:.3f} GiB')
if cache_rate > 0:
    print(f'with cache ({int(cache_rate*100)}% of data)        : {total_cpu_with_cache:.3f} GiB')
print()

# Recommendations
print('=== Practical Recommendations ===')
print(f'Minimum safe CPU RAM: {max(total_cpu_base, 16):.1f} GiB')
print(f'Recommended CPU RAM: {total_cpu_with_cache * 1.5:.1f} GiB (with 50% headroom)')
print()
print('Note: Update estimated_total_samples to your actual dataset size for more accurate cache estimate.')


=== CPU Memory Estimator for Data Loading ===
batch_size                           : 10
patch_size (per sample)              : [128, 128, 128]
view_size (per forward)              : [64, 64, 64]
num_workers (DataLoader)              : 10
pin_memory                           : True
cache_rate                          : 0.0

--- One full batch in CPU (before GPU transfer) ---
one patch batch (64x64x64 view)     : 0.078 GiB

--- DataLoader worker buffers ---
memory per worker (1 sample)         : 0.008 GiB
total 10 workers                     : 0.078 GiB

--- Pinned memory for async GPU transfer ---
pinned buffer size (2x batch)         : 0.156 GiB

--- CacheDataset ---
cache_rate=0 (streaming mode, no caching)

--- Transform pipeline overhead ---
estimated intermediate tensors      : 0.156 GiB

=== CPU Memory Summary ===
base (batch+workers+pinned+xforms)  : 0.469 GiB

=== Practical Recommendations ===
Minimum safe CPU RAM: 16.0 GiB
Recommended CPU RAM: 0.7 GiB (with 50% headroom)

Note: